# End-to-end pipeline and ablation study

Identifies which pipeline comparisons are supported by compatible paired data and audits recommendation lookup traceability.

## Controlled experiment availability

Disease rows have binary symptom vectors and labels. NER text rows have BIO spans but no disease labels or row-level linkage. A direct oracle-vs-predicted end-to-end score is therefore blocked; these data must not be compared as a controlled experiment.

In [1]:
from pathlib import Path
import sys
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'src').is_dir() and (p/'data').is_dir())
sys.path.insert(0,str(ROOT));sys.path.insert(0,str(ROOT/'results'))
from utils.evaluation import setup,save,meta
ROOT=setup(ROOT);OUT=ROOT/'results'
import pandas as pd,json
ner=pd.read_csv(ROOT/'data/ner/splits/ner_test.csv');disease=pd.read_csv(ROOT/'data/disease_prediction/Testing.csv')
status=pd.DataFrame([('GT NER symptoms → disease','BLOCKED','No paired disease labels for NER text test rows'),('Predicted NER → disease','BLOCKED','No compatible paired text-to-disease examples'),('Predicted plus normalized → disease','BLOCKED','No compatible paired labels'),('Disease classifier test split','AVAILABLE',f'{len(disease)} symptom-vector rows; see notebook 06')],columns=['experiment','status','reason']);display(status);status.to_csv(OUT/'tables/end_to_end_status.csv',index=False)


,experiment,status,reason
0,GT NER symptoms → disease,BLOCKED,No paired disease labels for NER text test rows
1,Predicted NER → disease,BLOCKED,No compatible paired text-to-disease examples
2,Predicted plus normalized → disease,BLOCKED,No compatible paired labels
3,Disease classifier test split,AVAILABLE,42 symptom-vector rows; see notebook 06


## Recommendation data traceability

Lookup key presence is evaluated, not clinical relevance or safety.

In [2]:
import re
prec=pd.read_csv(ROOT/'data/recommendations/symptom_precaution.csv'); tx=json.loads((ROOT/'models/recommendations/treatment_data.json').read_text());norm=lambda s:' '.join(re.sub(r'[^\w\s]',' ',str(s).lower()).split());pk={norm(s) for s in prec.Disease.dropna()};tk={norm(s) for s in tx}; names=sorted(disease.prognosis.astype(str).unique());cov=pd.DataFrame([{'disease':s,'precaution_found':norm(s) in pk,'treatment_found':norm(s) in tk} for s in names]);display(cov);cov.to_csv(OUT/'tables/recommendation_traceability.csv',index=False);print('precaution coverage',cov.precaution_found.mean(),'treatment coverage',cov.treatment_found.mean())


,disease,precaution_found,treatment_found
0,(vertigo) Paroymsal Positional Vertigo,True,True
1,AIDS,True,True
2,Acne,True,True
3,Alcoholic hepatitis,True,True
4,Allergy,True,True
5,Arthritis,True,True
6,Bronchial Asthma,True,True
7,Cervical spondylosis,True,True
8,Chicken pox,True,True
9,Chronic cholestasis,True,True


precaution coverage 1.0 treatment coverage 0.975609756097561
